# FIT5202 Data processing for Big data

##  Week 2 Lab: Manipulating Data with Spark DataFrames

Welcome back! Last week you got Spark running and touched your first DataFrame. This week, we'll build on that foundation and answer one important engineering question:

> **How do we manipulate large datasets using Spark DataFrames?**

> **Before you begin:** If you joined the unit this week, this notebook assumes your Docker container is already running and that you're comfortable with the basics from Week 1 (`SparkSession`, `show()`, `printSchema()`, `select()`, `filter()`). If any of that feels unfamiliar, it's worth a quick look back at last week's notebook first.


## Today's Plan

By the end of today's lab, you'll be able to:
1. read data from CSV, JSON and text files;
2. select, create, drop and rename columns;
3. filter rows using multiple conditions;
4. sort and limit results;
5. see what "Spark DataFrames are immutable" actually means;
6. chain several operations into one clean query;
7. observe *when* Spark actually does the work, and reflect on what makes a Spark DataFrame different from a pandas DataFrame; and
8. take home three more techniques to practise before next week.

### Notebook Shortcuts
<font color='blue'>
    <strong>Notebook shortcuts you need to be familiar with and use frequently:</strong>
    
- Run your cells using SHIFT+ENTER (or "Run cell")
- Run the current cell and insert a new cell below: ALT+ENTER
- To see more commands, please click the "menu" option (e.g. "Insert", "Cell")
- To see more keyboard shortcuts, click the above "keyboard image" button. Use "Esc" to enter command mode. Then, you can use a command. Some of the popular shortcuts are 
    - Basic navigation: enter, shift-enter, up/k, down/j
    - Saving the notebook: s
    - Cell creation: `a` = insert cell above, `b` = insert cell below
</font>
    
Let's get started.

## Table of Contents

1. [Quick Recap & Reading Data](#part-1)
2. [Working with Columns](#part-2)
3. [Filtering Rows](#part-3)
4. [Sorting and Limiting Rows](#part-4)
5. [Immutable DataFrames](#part-5)
6. [Method Chaining](#part-6)
7. [When Does Spark Actually Do the Work?](#part-7)
8. [Take-Home Practice](#take-home)
9. [Stop Spark](#stop-spark)


<a id="part-1"></a>
# Part 1 — Quick Recap & Reading Data

Let's start Spark the same way we did in Week 1.

In [3]:
# Import SparkConf
from pyspark import SparkConf
from pyspark.sql import SparkSession

master = "local[*]"
app_name = "FIT5202 - Week 2 - Manipulating Data with Spark DataFrames"

spark_conf = SparkConf().setMaster(master).setAppName(app_name)

spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("SparkSession created.")

SparkSession created.


### Reading data from files

So far you've mostly built DataFrames directly from Python lists. In practice, you'll almost always load data from a file instead. `spark.read` has a method for each common format.

**CSV** — `spark.read.csv(path, header=True, inferSchema=True)`. `header=True` tells Spark the first line contains column names; `inferSchema=True` asks Spark to guess sensible data types instead of treating every column as text.

More information on:
https://spark.apache.org/docs/latest/sql-data-sources-csv.html

In [4]:
bank_df = spark.read.csv(
    "resources/bank.csv",
    header=True,
    inferSchema=True
)

# Meet the data, the same four checks from Week 1.
bank_df.show(5)
bank_df.printSchema()
print("Column names:", bank_df.columns)
print("Number of rows:", bank_df.count())

+---+----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|age|       job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+---+----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
| 59|    admin.|married|secondary|     no|   2343|    yes|  no|unknown|  5|  may|    1042|       1|   -1|       0| unknown|    yes|
| 56|    admin.|married|secondary|     no|     45|     no|  no|unknown|  5|  may|    1467|       1|   -1|       0| unknown|    yes|
| 41|technician|married|secondary|     no|   1270|    yes|  no|unknown|  5|  may|    1389|       1|   -1|       0| unknown|    yes|
| 55|  services|married|secondary|     no|   2476|    yes|  no|unknown|  5|  may|     579|       1|   -1|       0| unknown|    yes|
| 54|    admin.|married| tertiary|     no|    184|     no|  no|unknown|  5| 

**JSON** — `spark.read.json(path)`. Each line of the file contains one JSON object; Spark builds the schema from the keys it finds. No `header` or `inferSchema` option needed — the structure is already in the file.

More information on: https://spark.apache.org/docs/latest/sql-data-sources-json.html

In [5]:
json_df = spark.read.json("resources/people.json")
json_df.show()
json_df.printSchema()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)



Notice `Michael`'s row: his record in the file never included an `age` key at all. Spark still needs one column per field, so it fills the gap with `null`. This is worth remembering — real-world JSON is often inconsistent record-to-record, and missing fields become `null`, not an error.

**Plain text** — `spark.read.text(path)`. This is different from the other two: it doesn't try to parse anything. Every line becomes one row in a single column called `value`, as raw text.

More information on: https://spark.apache.org/docs/latest/sql-data-sources-text.html

In [6]:
text_df = spark.read.text("resources/people.txt")
text_df.show(truncate=False)
text_df.printSchema()

+-----------+
|value      |
+-----------+
|Michael, 29|
|Andy, 30   |
|Justin, 19 |
+-----------+

root
 |-- value: string (nullable = true)



#### Lab Task 1
<a class="anchor" id="lab-task-1"></a>

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">1. Lab Task: </strong> Complete the code below to:

<ol>
    <li>read <b>resources/people.csv</b> as a Spark DataFrame, using <b>header=True</b> and the correct delimiter (<b>hint</b>: check the file, it isn't a comma!) and display it;</li>
    <li>read <b>resources/people.json</b>, display it, and print its schema; and</li>
    <li>read <b>resources/people.txt</b> and display it.</li>
</ol>

For each, also print how many rows it contains.

<strong>COMPLETE THE CODE BELOW.</strong>
</div>

In [7]:
# Lab Task 1

# 1. Read people.csv (semicolon-delimited) and display it
# YOUR ANSWER HERE
people_df = spark.read.csv(
    "resources/people.csv",
    header=True,
    inferSchema=True
)
print("people.csv:")
people_df.show()
people_df.printSchema()

# 2. Read people.json, display it, and print its schema
# YOUR ANSWER HERE
people_js_df = spark.read.csv(
    "resources/people.json",
    header=True,
    inferSchema=True
)
print("people.json:")
people_js_df.show()
people_js_df.printSchema()

# 3. Read people.txt and display it
# YOUR ANSWER HERE
people_tx_df = spark.read.text(
    "resources/people.txt"
)
print("people.txt:")
people_tx_df.show()
people_tx_df.printSchema()

people.csv:
+------------------+
|      name;age;job|
+------------------+
|Jorge;30;Developer|
|  Bob;32;Developer|
+------------------+

root
 |-- name;age;job: string (nullable = true)

people.json:
+------------------+
|{"name":"Michael"}|
+------------------+
|    {"name":"Andy"|
|  {"name":"Justin"|
+------------------+

root
 |-- {"name":"Michael"}: string (nullable = true)

people.txt:
+-----------+
|      value|
+-----------+
|Michael, 29|
|   Andy, 30|
| Justin, 19|
+-----------+

root
 |-- value: string (nullable = true)



<a id="part-2"></a>
# Part 2 — Working with Columns

We'll work with `bank_df` for the rest of today's lab. Last week you used `select()` to pick columns. Today we'll add: **aliasing**, **creating** a new column, **dropping** a column, and **renaming** a column.

`select()` : pick the columns you want, in any order.

In [8]:
bank_df.select("age", "job", "balance").show(5)

+---+----------+-------+
|age|       job|balance|
+---+----------+-------+
| 59|    admin.|   2343|
| 56|    admin.|     45|
| 41|technician|   1270|
| 55|  services|   2476|
| 54|    admin.|    184|
+---+----------+-------+
only showing top 5 rows


`.alias(new_name)` : give a column (or an expression) a temporary display name, without changing the DataFrame's actual columns. It's most useful inside `select()`.

In [9]:
from pyspark.sql.functions import col

bank_df.select(
    col("balance").alias("account_balance")
).show(5)

# bank_df itself still has "balance", not "account_balance", alias() only affects this select().
print("bank_df columns unchanged:", "balance" in bank_df.columns)

+---------------+
|account_balance|
+---------------+
|           2343|
|             45|
|           1270|
|           2476|
|            184|
+---------------+
only showing top 5 rows
bank_df columns unchanged: True


`withColumn(name, expression)` : actually create a new column on the DataFrame (unlike `alias()`, which only renames within one `select()`).

In [10]:
# withColumn() returns a NEW DataFrame: bank_df itself is untouched (more on this in Part 5).
demo_df = bank_df.withColumn("balance_in_thousands", col("balance") / 1000)
demo_df.select("balance", "balance_in_thousands").show(5)

+-------+--------------------+
|balance|balance_in_thousands|
+-------+--------------------+
|   2343|               2.343|
|     45|               0.045|
|   1270|                1.27|
|   2476|               2.476|
|    184|               0.184|
+-------+--------------------+
only showing top 5 rows


`drop(column_name)` : remove a column you don't need.

In [11]:
demo_df = bank_df.drop("contact")
print("Columns after dropping 'contact':", demo_df.columns)

Columns after dropping 'contact': ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'deposit']


`withColumnRenamed(old_name, new_name)` : rename a column, keeping every other column as-is.

In [12]:
demo_df = bank_df.withColumnRenamed("deposit", "subscribed")
print("Columns after renaming 'deposit':", demo_df.columns)

Columns after renaming 'deposit': ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'subscribed']


### Writing cleaner code: Method Chaining

Notice we've been reassigning `demo_df` in every example above. Once you need to combine several of these in a row, that pattern gets repetitive:

```python
demo_df = bank_df.withColumn("balance_in_thousands", col("balance") / 1000)
demo_df = demo_df.drop("contact")
demo_df = demo_df.withColumnRenamed("deposit", "subscribed")
```

This works perfectly well. However, once you start applying several transformations, the repeated reassignment can become difficult to read.

A more common Spark style is **method chaining**, where each operation is written on a new line:

```python
demo_df = (
    bank_df
    .withColumn("balance_in_thousands", col("balance") / 1000)
    .drop("contact")
    .withColumnRenamed("deposit", "subscribed")
)
```

Same result, easier to read. We'll use this style throughout the rest of the semester, please use it in the task below.

#### Lab Task 2
<a class="anchor" id="lab-task-2"></a>

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">2. Lab Task: </strong> Starting from <b>bank_df</b> (don't reassign <b>bank_df</b> itself, store your result in a new variable called <b>task2_df</b>), write <b>one chained expression</b> that:

<ol>
    <li>creates a new column called <b>balance_in_thousands</b>, equal to <b>balance / 1000</b>;</li>
    <li>drops the <b>day</b> and <b>month</b> columns;</li>
    <li>renames <b>deposit</b> to <b>subscribed</b>; and</li>
    <li>displays the resulting columns to confirm your changes.</li>
</ol>

<strong>COMPLETE THE CODE BELOW.</strong>
</div>

In [13]:
# Lab Task 2

# YOUR ANSWER HERE
task2_df = (
    bank_df
    .withColumn("balance_in_thousands", col("balance") / 1000)
    .drop("drop", "month")
    .withColumnRenamed("deposit", "subscribed")

)

# Display the resulting columns
# YOUR ANSWER HERE
task2_df.show(5)

+---+----------+-------+---------+-------+-------+-------+----+-------+---+--------+--------+-----+--------+--------+----------+--------------------+
|age|       job|marital|education|default|balance|housing|loan|contact|day|duration|campaign|pdays|previous|poutcome|subscribed|balance_in_thousands|
+---+----------+-------+---------+-------+-------+-------+----+-------+---+--------+--------+-----+--------+--------+----------+--------------------+
| 59|    admin.|married|secondary|     no|   2343|    yes|  no|unknown|  5|    1042|       1|   -1|       0| unknown|       yes|               2.343|
| 56|    admin.|married|secondary|     no|     45|     no|  no|unknown|  5|    1467|       1|   -1|       0| unknown|       yes|               0.045|
| 41|technician|married|secondary|     no|   1270|    yes|  no|unknown|  5|    1389|       1|   -1|       0| unknown|       yes|                1.27|
| 55|  services|married|secondary|     no|   2476|    yes|  no|unknown|  5|     579|       1|   -1| 

<a id="part-3"></a>
# Part 3 — Filtering Rows

In Week 1, you learned how to use `filter()` to select rows that satisfy a condition.

In this section, we'll extend that idea by learning how to:
- combine multiple conditions;
- identify missing values; and
- use a few convenient filtering methods.

### Combining multiple conditions

Use `&` (AND) when all conditions must be satisfied, and `|` (OR) when any condition can be satisfied.

Note: Each individual condition must be enclosed in parentheses.

In [14]:
# AND — both conditions must be true
bank_df.filter((bank_df["age"] > 40) & (bank_df["balance"] > 2000)).show(5)

# OR — either condition can be true
bank_df.filter((bank_df["job"] == "management") | (bank_df["job"] == "admin.")).show(5)

+---+----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|age|       job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+---+----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
| 59|    admin.|married|secondary|     no|   2343|    yes|  no|unknown|  5|  may|    1042|       1|   -1|       0| unknown|    yes|
| 55|  services|married|secondary|     no|   2476|    yes|  no|unknown|  5|  may|     579|       1|   -1|       0| unknown|    yes|
| 43|management| single| tertiary|     no|   2067|    yes|  no|unknown|  8|  may|     756|       1|   -1|       0| unknown|    yes|
| 52|management|married|  unknown|     no|   2240|    yes|  no|unknown| 13|  may|     845|       1|   -1|       0| unknown|    yes|
| 43|technician|married|secondary|     no|   3285|    yes|  no|unknown| 13| 

### Working with missing values

Spark provides two built-in methods for identifying missing values.
- isNull() — selects rows containing NULL values.
- isNotNull() — selects rows that are not NULL.

The `bank_df` dataset does not contain true NULL values, so let's demonstrate these methods using a small example DataFrame.

In [15]:
sample_data = [("Alice", 34), ("Bob", None), ("Charlie", 25)]
sample_df = spark.createDataFrame(sample_data, ["name", "age"])

sample_df.filter(sample_df["age"].isNull()).show()
sample_df.filter(sample_df["age"].isNotNull()).show()

+----+----+
|name| age|
+----+----+
| Bob|NULL|
+----+----+

+-------+---+
|   name|age|
+-------+---+
|  Alice| 34|
|Charlie| 25|
+-------+---+



### Missing values are not always NULL

In real datasets, missing information is not always stored as ``NULL``.

For example, ``bank_df`` represents missing values using the string ``"unknown"``.

You can verify this by running:
`bank_df.select("education").distinct().show()` 

Since `"unknown"` is simply a string rather than a NULL value, you filter it in the same way as any other value:

```python
bank_df.filter(bank_df["job"] != "unknown")
```

### Additional filtering methods
Spark also provides several convenient methods for common filtering tasks.

- **`isin([...])`** — keeps rows where a column matches one of several specified values.
- **`between(low, high)`** — keeps rows whose values fall within an inclusive numeric or date range.
- **`like("pattern")`** — performs SQL-style pattern matching, where % represents any sequence of characters.

In [16]:
# Example: keep customers aged between 25 and 40
bank_df.filter(bank_df["age"].between(25, 40)).show(5)

# Try these next:
# isin() keeps rows where a value matches one of several choices.
# Example:
bank_df.filter(bank_df["job"].isin("admin.", "technician", "management")).show(5)

# like() keeps rows where text matches a pattern.
# "%tech%" means the text can contain anything before or after "tech".
# Example:
bank_df.filter(bank_df["job"].like("%tech%")).show(5)

+---+-----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|age|        job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+---+-----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
| 37| technician|married|secondary|     no|      1|    yes|  no|unknown|  6|  may|     608|       1|   -1|       0| unknown|    yes|
| 28|   services| single|secondary|     no|   5090|    yes|  no|unknown|  6|  may|    1297|       3|   -1|       0| unknown|    yes|
| 38|     admin.| single|secondary|     no|    100|    yes|  no|unknown|  7|  may|     786|       1|   -1|       0| unknown|    yes|
| 30|blue-collar|married|secondary|     no|    309|    yes|  no|unknown|  7|  may|    1574|       2|   -1|       0| unknown|    yes|
| 29| management|married| tertiary|     no|    199|    yes| yes|unkno

#### Lab Task 3
<a class="anchor" id="lab-task-3"></a>

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">3. Lab Task: </strong> Using <b>bank_df</b>, complete the code below to:

<ol>
    <li>display customers older than 40 <b>and</b> with a balance greater than 2000;</li>
    <li>display customers whose <b>job</b> is not the placeholder value <b>"unknown"</b>; and</li>
    <li>display customers using <b>one</b> of <b>isin()</b>, <b>between()</b>, or <b>like()</b> — your choice of column and condition.</li>
</ol>

<strong>COMPLETE THE CODE BELOW.</strong>
</div>

In [17]:
# Lab Task 3

# 1. Older than 40 AND balance greater than 2000
# YOUR ANSWER HERE
bank_df.filter(bank_df["balance"].between(40, 2000)).show(5)

# 2. Job is not "unknown"
# YOUR ANSWER HERE
bank_df.filter(bank_df["job"]!="unknown").show(5)

# 3. Your choice: isin(), between(), or like()
# YOUR ANSWER HERE
bank_df.filter(bank_df["education"].like("%sec%")).show(5)

+---+----------+--------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|age|       job| marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+---+----------+--------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
| 56|    admin.| married|secondary|     no|     45|     no|  no|unknown|  5|  may|    1467|       1|   -1|       0| unknown|    yes|
| 41|technician| married|secondary|     no|   1270|    yes|  no|unknown|  5|  may|    1389|       1|   -1|       0| unknown|    yes|
| 54|    admin.| married| tertiary|     no|    184|     no|  no|unknown|  5|  may|     673|       2|   -1|       0| unknown|    yes|
| 56|management| married| tertiary|     no|    830|    yes| yes|unknown|  6|  may|    1201|       1|   -1|       0| unknown|    yes|
| 60|   retired|divorced|secondary|     no|    545|    yes|  no|unkno

<a id="part-4"></a>
# Part 4 — Sorting and Limiting Rows

`orderBy()` sorts rows by one or more columns. `limit()` keeps only the first *n* rows of whatever you currently have.

In [18]:
# Ascending (the default)
bank_df.orderBy("age").show(5)

# Descending
bank_df.orderBy(bank_df["balance"].desc()).show(5)

# limit() keeps only the first n rows
bank_df.orderBy(bank_df["balance"].desc()).limit(3).show()

+---+-------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+-------+
|age|    job|marital|education|default|balance|housing|loan| contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+---+-------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+-------+
| 18|student| single|  primary|     no|    608|     no|  no|cellular| 12|  aug|     267|       1|   -1|       0| unknown|    yes|
| 18|student| single|  unknown|     no|    108|     no|  no|cellular| 10|  aug|     167|       1|   -1|       0| unknown|    yes|
| 18|student| single|  unknown|     no|    108|     no|  no|cellular|  8|  sep|     169|       1|   -1|       0| unknown|    yes|
| 18|student| single|  primary|     no|    608|     no|  no|cellular| 13|  nov|     210|       1|   93|       1| success|    yes|
| 18|student| single|  unknown|     no|    108|     no|  no|cellular|  9|  feb|      92|  

#### Lab Task 4
<a class="anchor" id="lab-task-4"></a>

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">4. Lab Task: </strong> Using <b>bank_df</b>, complete the code below to:

<ol>
    <li>sort <b>bank_df</b> by <b>balance</b>, highest first;</li>
    <li>sort <b>bank_df</b> by <b>age</b>, youngest first, and display only the first 5 rows; and</li>
    <li>select <b>age</b>, <b>job</b> and <b>balance</b>, sort by <b>balance</b> highest first, and display only the top 10 rows.</li>
</ol>

<strong>COMPLETE THE CODE BELOW.</strong>
</div>

In [19]:
# Lab Task 4

# 1. Sort by balance, highest first
# YOUR ANSWER HERE
bank_df.orderBy(bank_df["balance"].desc()).show(5)

# 2. Sort by age, youngest first, first 5 rows only
# YOUR ANSWER HERE
bank_df.orderBy(bank_df["age"]).show(5)

# 3. Select age, job, balance; sort by balance descending; top 10 rows
# YOUR ANSWER HERE
bank_df.orderBy(bank_df["balance"].desc()).select("age", "job", "balance").show(5)

+---+-------------+--------+---------+-------+-------+-------+----+---------+---+-----+--------+--------+-----+--------+--------+-------+
|age|          job| marital|education|default|balance|housing|loan|  contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+---+-------------+--------+---------+-------+-------+-------+----+---------+---+-----+--------+--------+-----+--------+--------+-------+
| 84|      retired| married|secondary|     no|  81204|     no|  no|telephone|  1|  apr|     390|       1|   94|       3| success|    yes|
| 84|      retired| married|secondary|     no|  81204|     no|  no|telephone| 28|  dec|     679|       1|  313|       2|   other|    yes|
| 52|  blue-collar| married|  primary|     no|  66653|     no|  no| cellular| 14|  aug|     109|       3|   -1|       0| unknown|     no|
| 43|       admin.|  single|secondary|     no|  56831|     no|  no|  unknown| 15|  may|     243|       1|   -1|       0| unknown|     no|
| 61|self-employed|divorced| terti

<a id="part-5"></a>
# Part 5 — Immutable DataFrames

If you've used `pandas` before, you might expect the following line to remove everyone aged 30 or younger from `bank_df`:

Do you think it will?

```python
bank_df.filter(bank_df["age"] > 30)
```

Let's find out.

In [20]:
filtered = bank_df.filter(bank_df["age"] > 30)

print("bank_df row count:", bank_df.count())
print("filtered row count:", filtered.count())

bank_df row count: 11162
filtered row count: 9155


**What happened?**

Even though we called `filter()`, `bank_df` still has all 11,162 rows. That's because `filter()` didn't change `bank_df`. Instead, it created a new DataFrame, which we stored in the variable `filtered`.

This behaviour is called immutability.

**Spark DataFrames are immutable.** Every operation you've used today: `select()`, `filter()`, `withColumn()`, `drop()`, `orderBy()`, they works the same way: none of them modify the DataFrame they're called on, they all returns a new DataFrame, leaving the original one unchanged.

That's exactly why, throughout this notebook, we've had to write `bank_df = bank_df.drop(...)`. We're not changing the original DataFrame. We're simply telling `bank_df` to refer to the new DataFrame returned by `drop()`.

**Keep this in mind, it's going to matter again next week.**

<a id="part-6"></a>
# Part 6 — Method Chaining

You've already seen the chaining style in Part 2. Now that you know `filter()`, `select()`, and `orderBy()` too, let's put them all in one query.

Instead of:

```python
step1 = bank_df.filter(bank_df["balance"] > 2000)
step2 = step1.select("age", "job", "balance")
step3 = step2.orderBy(step2["balance"].desc())
step3.show(10)
```

write:

```python
(
    bank_df
    .filter(bank_df["balance"] > 2000)
    .select("age", "job", "balance")
    .orderBy(col("balance").desc())
    .show(10)
)
```

Same result. No intermediate variables, no repeated DataFrame names, and it reads as one sentence: *"take bank_df, filter it, select these columns, sort it, show 10 rows."*

#### Lab Task 5
<a class="anchor" id="lab-task-5"></a>

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">5. Lab Task: </strong> Write <b>one chained expression</b>, starting from <b>bank_df</b>, that:

<ol>
    <li>keeps only customers with a housing loan (<b>housing == "yes"</b>);</li>
    <li>selects <b>age</b>, <b>job</b>, <b>balance</b>, and <b>duration</b>;</li>
    <li>sorts by <b>duration</b>, highest first; and</li>
    <li>shows only the top 10 rows.</li>
</ol>

<strong>COMPLETE THE CODE BELOW.</strong>
</div>

In [21]:
# Lab Task 5
# YOUR ANSWER HERE
(
    bank_df
    .filter(bank_df["housing"] == "yes")
    .select("age", "job", "balance")
    .orderBy(bank_df["duration"].desc())
    .show(10)

)

+---+-------------+-------+
|age|          job|balance|
+---+-------------+-------+
| 36|self-employed|   -103|
| 53|       admin.|    849|
| 44|     services|     51|
| 47|  blue-collar|    238|
| 41|   management|   3234|
| 36|    housemaid|     78|
| 29|  blue-collar|    213|
| 47|  blue-collar|    126|
| 43|  blue-collar|   3064|
| 41|   technician|    650|
+---+-------------+-------+
only showing top 10 rows


<a id="part-7"></a>
# Part 7 — When Does Spark Actually Do the Work?

Before you run the chain you just wrote in Lab Task 5, did Spark actually filter, select, and sort anything the moment you called those methods?

Let's find out.

In [22]:
# Build a chain of operations. Nothing is computed yet, we're just describing what we want.
big_balances = (
    bank_df
    .filter(bank_df["balance"] > 5000)
    .select("age", "job", "balance")
    .orderBy(bank_df["balance"].desc())
)

print("Chain built.")

Chain built.


Open the Spark UI at **http://localhost:4040** and click the **Jobs** tab.

You should see... nothing. No job has run. Spark hasn't touched the data yet, it has only recorded *what* you asked for, not done it.

Operations like `filter()`, `select()`, `withColumn()`, and `orderBy()` are called **transformations**. A transformation describes a step, but doesn't run it.

### Think Before You Run

So far, you've built a chain of transformations, but nothing has appeared in the Spark UI. That suggests Spark hasn't actually processed the data yet.

Before you run the next cell, make a prediction:

**A.** `.show()` will also do nothing, it's just one more step being added to the chain.

**B.** `.show()` will finally tell Spark to run the whole chain, and a job will appear in the Spark UI.

Write down your guess. Then run the cell below and check the Spark UI again.

In [23]:
# Your guess?
# A or B

big_balances.show(10)

+---+-------------+-------+
|age|          job|balance|
+---+-------------+-------+
| 84|      retired|  81204|
| 84|      retired|  81204|
| 52|  blue-collar|  66653|
| 43|       admin.|  56831|
| 61|self-employed|  52587|
| 61|self-employed|  52587|
| 56| entrepreneur|  51439|
| 39|   technician|  45248|
| 75|      retired|  37127|
| 51| entrepreneur|  36935|
+---+-------------+-------+
only showing top 10 rows


**What happened?**
Now go back to the Spark UI, and refresh the page. 

**Did a job appear this time?**

If so, you've just seen one of Spark's most important ideas.


`.show()` is an **action**, Unlike `filter()`, `select()`, or `orderBy()`, an action tells Spark: **"Now go and compute the result."** Everything you've chained together beforehand is executed at that point.

Other common actions include: `.count()`, `.collect()`, `.first()`, `.take()`, and `writing data to a file or database`. Until Spark reaches an action, it simply records the transformations you've requested.

This behaviour is called **lazy evaluation**.

**Let's connect the ideas** 

Notice something interesting.

In **Part 5**, you learned that Spark DataFrames are **immutable**, every operation creates a new DataFrame instead of changing the original one.

Now you've discovered something else: **Spark also waits until an action is called before it actually performs the computation.**

These two ideas, immutability and lazy evaluation, are two of the most important concepts in Spark.

A pattern should be emerging: 
> Spark doesn't do any work until it absolutely has to, and even then, it never modifies your original DataFrame.

Behind the scenes, Spark turns your whole chain into an execution plan before running it, rather than executing them one line at a time.

Next week, when we look at parallel search and partitioning, we'll open that plan up and see exactly how Spark uses it to split the work across your machine.

# Reflection

Today we manipulated data using **Spark DataFrames**. You'll probably notice that many of the operations look very similar to those in pandas.

Before moving on, take a moment to reflect:
> **Based on today's lab, what are at least two differences between a Spark DataFrame and a pandas DataFrame?**

Hints:
- Where is the data stored?
- When is the computation executed?
- Can the DataFrame be modified directly?

💡 There isn't a single correct answer. The goal is to think about how Spark behaves differently from the tools you may already know.

<a id="take-home"></a>
# Take Home Practice

Today's lab introduced the core DataFrame operations you'll use throughout this unit.

The following three exercises build on the same ideas and introduce a few additional techniques that you'll find useful in later labs and assignments.

Work through them using `bank_df`, and take your time to understand what each operation is doing.

As you work, remember to check your results frequently, for example, use `printSchema()` after casting a column or `show()` after creating a new one, to make sure each step behaves as expected.

### 1. Casting data types

CSV files with `inferSchema=True` don't always guess the type you want. `col(...).cast("type")` lets you convert a column to a specific type — common ones are `"int"`, `"float"`, `"double"`, and `"string"`.

**Try it:** cast `balance` to a `float`, storing the result as a new column called `balance_float`. Confirm it worked with `printSchema()`.

In [35]:
# YOUR ANSWER HERE
balance_float_df = bank_df.withColumn(
    "balance_float",
    col("balance").cast("float")

)
balance_float_df.printSchema()

root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- balance: integer (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- deposit: string (nullable = true)
 |-- balance_float: float (nullable = true)



### 2. String operations

`pyspark.sql.functions` has string helpers that work the same way as `withColumn()` — `upper()`, `lower()`, `concat()`, and `substring()` are the common ones.

**Try it:**
1. Create `job_upper`, an uppercase version of `job` (hint: `upper(col("job"))`).
2. Create `summary`, combining `job` and `marital` into one string, separated by `" - "` (hint: `concat(col("job"), lit(" - "), col("marital"))` — `lit()` inserts a literal value, like the separator, into the expression).

In [38]:
from pyspark.sql.functions import upper, concat, lit

# YOUR ANSWER HERE

string_df = (
    bank_df
    .withColumn(
    "job_upper",
    upper(col("job")))
    
    .withColumn(
    "summary",
    concat(col("job"), lit("-"), col("marital")))
)

string_df.printSchema()

root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- balance: integer (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- deposit: string (nullable = true)
 |-- job_upper: string (nullable = true)
 |-- summary: string (nullable = true)



### 3. Conditional columns with `when()`

`when()` works like an if/elif/else, but for a column. Chain `.when()` calls for each condition, and finish with `.otherwise()` for anything that doesn't match:

```python
df.withColumn(
    "new_column",
    when(condition_1, "value_1")
    .when(condition_2, "value_2")
    .otherwise("value_3")
)
```

**Try it:** create a column called `balance_level`:
- `"low"` if `balance` is below 0
- `"medium"` if `balance` is between 0 and 5000
- `"high"` if `balance` is 5000 or above

Then check your work: group by `balance_level` isn't something you've learned yet, so instead just `.filter()` for each level and eyeball a few rows to confirm the boundaries are right.

In [47]:
from pyspark.sql.functions import when

# YOUR ANSWER HERE

balance_level_df = bank_df.withColumn(
    "balance_level",
    when(col("balance") < 0, "low")
    .when(col("balance") < 5000, "medium" )
    .when(col("balance") >= 5000, "high")

)

balance_level_df.filter(
    col("balance_level") == "low"
).select("balance", "balance_level").show(5)

balance_level_df.filter(
    col("balance_level") == "medium"
).select("balance", "balance_level").show(5)

balance_level_df.filter(
    col("balance_level") == "high"
).select("balance", "balance_level").show(5)

+-------+-------------+
|balance|balance_level|
+-------+-------------+
|     -8|          low|
|   -192|          low|
|     -1|          low|
|   -395|          low|
|     -1|          low|
+-------+-------------+
only showing top 5 rows
+-------+-------------+
|balance|balance_level|
+-------+-------------+
|   2343|       medium|
|     45|       medium|
|   1270|       medium|
|   2476|       medium|
|    184|       medium|
+-------+-------------+
only showing top 5 rows
+-------+-------------+
|balance|balance_level|
+-------+-------------+
|   5090|         high|
|   7180|         high|
|   5291|         high|
|  10576|         high|
|   5773|         high|
+-------+-------------+
only showing top 5 rows


<a id="stop-spark"></a>
## Stop Spark

Run the next cell only when you have completely finished the notebook.

Stopping Spark releases the resources used by the active `SparkSession`.

In [48]:
spark.stop()
print(
    "SparkSession stopped. Great work manipulating DataFrames this week — "
    "see you next week, when we look at how Spark actually executes all of this."
)

SparkSession stopped. Great work manipulating DataFrames this week — see you next week, when we look at how Spark actually executes all of this.
